In [1]:
pip install wikipedia-api

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia-api: filename=wikipedia_api-0.9.0-py3-none-any.whl size=15582 sha256=7fe0e75f47326f91c448507d87e666c9c2c4c5a8ff62ed4474ace9c0ea22e44b
  Stored in directory: c:\users\aksha\appdata\local\pip\cache\wheels\23\31\e6\2f9ecaa7352d64bb8170b216d5adad1198a061e0f3fda073b0
Successfully built wikipedia-api
Note: you may need to restart the kernel to use updated packages.



#Initialize Wikipedia API



In [14]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(
    user_agent='Wikipedia_Hybrid_Rag_Project',
    language="en",
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

In [ ]:
# p_wiki = wiki.page("Artificial_intelligence")
# print(p_wiki.text)

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of cutting edge AI has filtered into general applications, often without being calle

#Helper Function – Validate Article Length

In [4]:
def is_valid_article(page, min_words=200):
    if not page.exists():
        return False
    text = page.text
    word_count = len(text.split())
    return word_count >= min_words


#Collect Fixed URLs (200 URLs)

In [5]:
seed_topics = [
    "Artificial_intelligence",
    "Machine_learning",
    "Physics",
    "Chemistry",
    "Biology",
    "History",
    "Geography",
    "Economics",
    "Psychology",
    "Sociology",
    "Mathematics",
    "Computer_science",
    "Philosophy",
    "Political_science",
    "Medicine",
    "Astronomy",
    "Environmental_science",
    "Engineering",
    "Linguistics",
    "Anthropology"
]

In [6]:
fixed_urls = set()

for topic in seed_topics:
    page = wiki.page(topic)
    if is_valid_article(page):
        fixed_urls.add(page.fullurl)

    # Expand via linked pages
    for link_title in list(page.links.keys()):
        if len(fixed_urls) >= 200:
            break
        linked_page = wiki.page(link_title)
        if is_valid_article(linked_page):
            fixed_urls.add(linked_page.fullurl)

    if len(fixed_urls) >= 200:
        break


In [7]:
print(len(fixed_urls))
print(fixed_urls)

200
{'https://en.wikipedia.org/wiki/Animal_machine', 'https://en.wikipedia.org/wiki/Neural_network_(machine_learning)', 'https://en.wikipedia.org/wiki/BBC_News', 'https://en.wikipedia.org/wiki/Basic_Books', 'https://en.wikipedia.org/wiki/Bayes_estimator', 'https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare', 'https://en.wikipedia.org/wiki/Bayesian_inference', 'https://en.wikipedia.org/wiki/2024_Indian_general_election', 'https://en.wikipedia.org/wiki/Google_AdSense', 'https://en.wikipedia.org/wiki/Research', 'https://en.wikipedia.org/wiki/Alternative_hypothesis', 'https://en.wikipedia.org/wiki/ACM_Computing_Classification_System', 'https://en.wikipedia.org/wiki/Agricultural_robot', 'https://en.wikipedia.org/wiki/AlexNet', 'https://en.wikipedia.org/wiki/Artificial_intelligence_in_government', 'https://en.wikipedia.org/wiki/Alberto_Ciaramella', 'https://en.wikipedia.org/wiki/Artificial_intelligence_in_Wikimedia_projects', 'https://en.wikipedia.org/wiki/Anti-realism', 'ht

#Save URL to JSON

In [8]:
import json

with open("data/fixed_urls.json", "w") as f:
    json.dump(list(fixed_urls), f, indent=2)


#Collect Random URLs (300 URLs)

In [16]:
HEADERS = {
    "User-Agent": "HybridRAG-StudentProject",
    "Accept": "application/json"
}


In [20]:
import requests

def get_random_titles(batch_size=10):
    response = requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/random/title?count={batch_size}",
        headers=HEADERS,
        timeout=10
    )
    print(response.status_code)
    if response.status_code != 200:
        print("Failed with status:", response.status_code)
        return []

    return [item["title"] for item in response.json()["items"]]



In [21]:
def is_valid_article_fast(wiki, title, min_summary_words=50, min_full_words=200):
    page = wiki.page(title)
    if not page.exists():
        return None

    # FAST pre-filter (summary)
    if len(page.summary.split()) < min_summary_words:
        return None

    # FINAL validation (full text)
    if len(page.text.split()) < min_full_words:
        return None

    return page


#FINAL optimized collection loop

In [ ]:
random_urls = set()
fixed_urls_set = set(fixed_urls)

MAX_ATTEMPTS = 2000
attempts = 0

while len(random_urls) < 300 and attempts < MAX_ATTEMPTS:
    attempts += 1
    titles = get_random_titles(batch_size=10)

    for title in titles:
        if len(random_urls) >= 300:
            break

        page = is_valid_article_fast(wiki, title)
        if page is None:
            continue

        if page.fullurl not in fixed_urls_set:
            random_urls.add(page.fullurl)

print(f"Collected {len(random_urls)} random URLs in {attempts} attempts")


In [23]:
print(random_urls)

{'https://en.wikipedia.org/wiki/GTFS', 'https://en.wikipedia.org/wiki/Lake_surfing', 'https://en.wikipedia.org/wiki/Buckley,_Greater_Manchester', 'https://en.wikipedia.org/wiki/Paul_Burlison', 'https://en.wikipedia.org/wiki/Independent_Police_Conduct_Commission', 'https://en.wikipedia.org/wiki/Meta-Object_Facility', 'https://en.wikipedia.org/wiki/Yubileyny_Sports_Palace', 'https://en.wikipedia.org/wiki/The_Monster_That_Challenged_the_World', 'https://en.wikipedia.org/wiki/San_Marino_and_Rimini_Riviera_motorcycle_Grand_Prix', 'https://en.wikipedia.org/wiki/Watling_Lodge', 'https://en.wikipedia.org/wiki/Oregon_Route_86', 'https://en.wikipedia.org/wiki/DMTN', 'https://en.wikipedia.org/wiki/Castle_Heights', 'https://en.wikipedia.org/wiki/Rock_gong', 'https://en.wikipedia.org/wiki/Leung_So_Kee', 'https://en.wikipedia.org/wiki/Don_Cherry_(trumpeter)', 'https://en.wikipedia.org/wiki/Discourse_on_Defilement', 'https://en.wikipedia.org/wiki/Gobernador_Pi%C3%B1ero,_San_Juan,_Puerto_Rico', 'https

In [ ]:
# import json

with open("data/random_urls.json", "w") as f:
    json.dump(list(random_urls), f, indent=2)


#📘 STEP 2: Extract & Clean Wikipedia Article Text

In [9]:
import json

with open("data/fixed_urls.json") as f:
    fixed_urls = json.load(f)

with open("data/random_urls.json") as f:
    random_urls = json.load(f)

all_urls = fixed_urls + random_urls
print("Total URLs:", len(all_urls))  # should be 500


Total URLs: 500


In [10]:
from urllib.parse import unquote

def get_title_from_url(url):
    return unquote(url.split("/wiki/")[-1])


In [12]:
def clean_wiki_text(text):
    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Stop at references or external links
        if line.lower().startswith("== references"):
            break
        if line.lower().startswith("== external links"):
            break

        cleaned_lines.append(line)

    return " ".join(cleaned_lines)


In [16]:
articles = []

for url in all_urls:
    title = get_title_from_url(url)
    page = wiki.page(title)

    if not page.exists():
        continue

    raw_text = page.text
    print(raw_text[:100])
    cleaned_text = clean_wiki_text(raw_text)
    print(cleaned_text[:100])

    # Enforce ≥200-word rule again (safety)
    if len(cleaned_text.split()) < 200:
        continue

    articles.append({
        "title": page.title,
        "url": page.fullurl,
        "text": cleaned_text
    })

print("Valid articles collected:", len(articles))


Animal machine (French: bête-machine), also known as animal automatism, is a philosophical concept m
Animal machine (French: bête-machine), also known as animal automatism, is a philosophical concept m
In machine learning, a neural network (NN) or neural net, also called an artificial neural network (
In machine learning, a neural network (NN) or neural net, also called an artificial neural network (
BBC News is an operational business division of the British Broadcasting Corporation (BBC) responsib
BBC News is an operational business division of the British Broadcasting Corporation (BBC) responsib
Basic Books is a book publisher founded in 1950 and located in New York City, now an imprint of Hach
Basic Books is a book publisher founded in 1950 and located in New York City, now an imprint of Hach
In estimation theory and decision theory, a Bayes estimator or a Bayes action is an estimator or dec
In estimation theory and decision theory, a Bayes estimator or a Bayes action is an estimat

In [17]:
with open("data/articles.json", "w") as f:
    json.dump(articles, f, indent=2)


#📦 STEP 3: Text Chunking (200–400 tokens, 50 overlap)

In [18]:
pip install transformers tqdm

Note: you may need to restart the kernel to use updated packages.


#STEP 3.1: Load Clean Articles

In [19]:
import json

with open("data/articles.json") as f:
    articles = json.load(f)

print("Articles loaded:", len(articles))


Articles loaded: 499


STEP 3.2: Initialize Tokenizer (for Token-Accurate Chunking)

We’ll use a SentenceTransformer-compatible tokenizer.
Why this matters

-Chunk size must be in tokens, not characters

-Keeps chunking aligned with embedding model

In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

STEP 3.3: Token-Based Chunking Function (Core Logic)

This uses a sliding window approach.

✔ Why this is correct
Uses tokens (assignment requirement)

Sliding window ensures overlap

Prevents loss of context at boundaries

In [22]:
def chunk_text(
    text,
    tokenizer,
    chunk_size=300,
    overlap=50
):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []

    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        print("chunk_tokens",chunk_tokens)

        chunk_text = tokenizer.decode(chunk_tokens)
        print("chunk_text",chunk_tokens)
        chunks.append(chunk_text)

        start += chunk_size - overlap

    return chunks


#STEP 3.4: Generate Chunks with Metadata & IDs

In [ ]:
from tqdm import tqdm

all_chunks = []
chunk_counter = 0

for article in tqdm(articles):
    text = article["text"]
    title = article["title"]
    url = article["url"]

    chunks = chunk_text(text, tokenizer)

    for chunk in chunks:
        # Optional: skip extremely small chunks
        if len(tokenizer.encode(chunk)) < 100:
            continue

        all_chunks.append({
            "chunk_id": f"chunk_{chunk_counter}",
            "title": title,
            "url": url,
            "text": chunk
        })
        chunk_counter += 1

print("Total chunks created:", len(all_chunks))


In [ ]:
print(all_chunks[0])

{'chunk_id': 'chunk_0', 'title': 'Animal machine', 'url': 'https://en.wikipedia.org/wiki/Animal_machine', 'text': "animal machine ( french : bete - machine ), also known as animal automatism, is a philosophical concept most closely associated with 17th - century philosopher rene descartes, who argued that non - human animals are automata — complex, self - moving biological machines devoid of thought, reason, consciousness, or immaterial souls. as part of his philosophy of mind – body dualism, descartes drew a sharp metaphysical boundary between humans, who he saw as possessing immaterial minds capable of rational thought and language, and animals, whose behaviors he attributed entirely to mechanical processes. these included the arrangement of physical organs, the flow of animal spirits, and responses to external stimuli, all governed by the same laws of motion that apply to inanimate matter. developed in opposition to the prevailing aristotelian and scholastic view that living beings 

STEP 3.5: Save Final Chunks to JSON

In [25]:
with open("data/chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

In [26]:
lengths = [len(tokenizer.encode(c["text"])) for c in all_chunks]

print("Min tokens:", min(lengths))
print("Max tokens:", max(lengths))
print("Avg tokens:", sum(lengths)/len(lengths))


Min tokens: 101
Max tokens: 305
Avg tokens: 294.1723163841808


PART1 - Step 4

🧠 STEP 4: Dense Vector Retrieval (Embeddings + FAISS)

In [ ]:
pip install sentence-transformers faiss-cpu numpy

STEP 4.1: Load Chunked Data

In [2]:
import json

with open("data/chunks.json") as f:
    chunks = json.load(f)

print("Chunks loaded:", len(chunks))


Chunks loaded: 5310


STEP 4.2: Load Sentence Embedding Model

In [24]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


In [6]:
chunk_texts = [chunk["text"] for chunk in chunks]

STEP 4.4: Generate Embeddings (CRITICAL STEP)

In [7]:
import numpy as np

embeddings = model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True   # VERY IMPORTANT
)

embeddings = np.array(embeddings).astype("float32")
print("Embedding shape:", embeddings.shape)


Batches:   0%|          | 0/166 [00:00<?, ?it/s]

Embedding shape: (5310, 384)


STEP 4.5: Build FAISS Index

In [26]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

index.add(embeddings)
print("Total vectors indexed:", index.ntotal)


NameError: name 'embeddings' is not defined

STEP 4.6: Save FAISS Index to Disk

In [9]:
import os

os.makedirs("indexes", exist_ok=True)

faiss.write_index(index, "indexes/faiss.index")


STEP 4.7: Save Metadata Mapping (VERY IMPORTANT)

FAISS only stores vectors — you must store metadata yourself.

In [10]:
metadata = []

for i, chunk in enumerate(chunks):
    metadata.append({
        "faiss_id": i,
        "chunk_id": chunk["chunk_id"],
        "title": chunk["title"],
        "url": chunk["url"],
        "text": chunk["text"]
    })

with open("indexes/faiss_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)


STEP 4.8: Dense Retrieval Function (Query-Time)

In [11]:
def dense_retrieve(query, model, index, metadata, top_k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(ids[0]):
        results.append({
            "rank": rank + 1,
            "score": float(scores[0][rank]),
            "chunk_id": metadata[idx]["chunk_id"],
            "title": metadata[idx]["title"],
            "url": metadata[idx]["url"],
            "text": metadata[idx]["text"]
        })

    return results


STEP 4.9: Test Dense Retrieval (MANDATORY)

In [14]:
results = dense_retrieve(
    query="What is self-attention in transformers?",
    model=model,
    index=index,
    metadata=metadata,
    top_k=5
)

for r in results:
    print(r["rank"], r["score"], r["title"])


1 0.5471163988113403 Attention (machine learning)
2 0.5217794179916382 Attention (machine learning)
3 0.5162132382392883 Attention (machine learning)
4 0.4935230612754822 Attention (machine learning)
5 0.492458701133728 Attention (machine learning)


🔎 STEP 5: Sparse Keyword Retrieval (BM25)

In [1]:
pip install rank-bm25 nltk

Note: you may need to restart the kernel to use updated packages.


STEP 5.1: Load Chunked Data

In [2]:
import json

with open("data/chunks.json") as f:
    chunks = json.load(f)

print("Chunks loaded:", len(chunks))


Chunks loaded: 5310


STEP 5.2: Text Preprocessing for BM25 (IMPORTANT)

In [15]:
import re

STOPWORDS = {"of", "the", "is", "and", "to", "in"}

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return tokens



STEP 5.3: Build BM25 Index

In [16]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [
    preprocess_text(chunk["text"]) for chunk in chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index built")

BM25 index built


STEP 5.4: BM25 Retrieval Function (Query-Time)

In [17]:
def bm25_retrieve(query, chunks, bm25, top_k=5):
    query_tokens = preprocess_text(query)
    scores = bm25.get_scores(query_tokens)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []
    for rank, idx in enumerate(ranked_indices):
        results.append({
            "rank": rank + 1,
            "score": float(scores[idx]),
            "chunk_id": chunks[idx]["chunk_id"],
            "title": chunks[idx]["title"],
            "url": chunks[idx]["url"],
            "text": chunks[idx]["text"]
        })

    return results


STEP 5.5: Test BM25 Retrieval (MANDATORY)

In [18]:
results = bm25_retrieve(
    query="Eiffel Tower Paris",
    chunks=chunks,
    bm25=bm25,
    top_k=5
)

for r in results:
    print(r["rank"], r["score"], r["title"])


1 15.486794753627166 Bertrand Russell
2 14.251207240255429 Aqueduc de Louveciennes
3 13.926318707446022 AI safety
4 13.55827689375661 AI safety
5 9.886467507287742 Large numbers


STEP 6.2: Implement RRF Function (CORE LOGIC)

In [19]:
def reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    k=60
):
    rrf_scores = {}

    # Dense retriever contribution
    for result in dense_results:
        chunk_id = result["chunk_id"]
        rank = result["rank"]
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank)

    # BM25 retriever contribution
    for result in bm25_results:
        chunk_id = result["chunk_id"]
        rank = result["rank"]
        rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank)

    # Sort by RRF score (descending)
    fused = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return fused


STEP 6.3: Merge Metadata Back (IMPORTANT)

In [20]:
def build_chunk_lookup(chunks):
    return {chunk["chunk_id"]: chunk for chunk in chunks}


Combine everything into final results

In [21]:
def build_final_rrf_results(
    fused_ranks,
    dense_results,
    bm25_results,
    chunk_lookup,
    top_n=5
):
    dense_rank_map = {r["chunk_id"]: r["rank"] for r in dense_results}
    bm25_rank_map = {r["chunk_id"]: r["rank"] for r in bm25_results}

    final_results = []

    for chunk_id, rrf_score in fused_ranks[:top_n]:
        chunk = chunk_lookup[chunk_id]

        final_results.append({
            "chunk_id": chunk_id,
            "title": chunk["title"],
            "url": chunk["url"],
            "text": chunk["text"],
            "rrf_score": rrf_score,
            "dense_rank": dense_rank_map.get(chunk_id),
            "bm25_rank": bm25_rank_map.get(chunk_id)
        })

    return final_results


STEP 6.4: End-to-End Hybrid Retrieval Function

In [58]:
def hybrid_retrieve(
    query,
    dense_model,
    faiss_index,
    faiss_metadata,
    bm25,
    chunks,
    top_k=10,
    top_n=5
):
    # Dense retrieval
    dense_results = dense_retrieve(
        query,
        dense_model,
        faiss_index,
        faiss_metadata,
        top_k=top_k
    )

    print("Dense results:", dense_results)

    # BM25 retrieval
    bm25_results = bm25_retrieve(
        query,
        chunks,
        bm25,
        top_k=top_k
    )

    print("BM25 results:", bm25_results)

    # RRF fusion
    fused = reciprocal_rank_fusion(dense_results, bm25_results, k=60)

    # Metadata
    chunk_lookup = build_chunk_lookup(chunks)

    final_results = build_final_rrf_results(
        fused,
        dense_results,
        bm25_results,
        chunk_lookup,
        top_n=top_n
    )

    return final_results


STEP 6.5: Test RRF (VERY IMPORTANT)

In [27]:
import faiss

index = faiss.read_index("indexes/faiss.index")
print("FAISS index loaded. Total vectors:", index.ntotal)


FAISS index loaded. Total vectors: 5310


In [28]:
import json

with open("indexes/faiss_metadata.json") as f:
    faiss_metadata = json.load(f)

print("Metadata loaded:", len(faiss_metadata))


Metadata loaded: 5310


In [38]:
results = hybrid_retrieve(
    query="How does the transformer architecture handle long-range dependencies in sequences?",
    dense_model=model,
    faiss_index=index,      # <-- NOW DEFINED
    faiss_metadata=faiss_metadata,
    bm25=bm25,
    chunks=chunks,
    top_k=10,
    top_n=5
)

for r in results:
    print(
        f"RRF={r['rrf_score']:.4f}",
        f"DenseRank={r['dense_rank']}",
        f"BM25Rank={r['bm25_rank']}",
        r["title"]
    )


RRF=0.0325 DenseRank=2 BM25Rank=1 Attention (machine learning)
RRF=0.0310 DenseRank=3 BM25Rank=6 Attention (machine learning)
RRF=0.0305 DenseRank=6 BM25Rank=5 Ashish Vaswani
RRF=0.0164 DenseRank=1 BM25Rank=None AlphaFold
RRF=0.0161 DenseRank=None BM25Rank=2 Autoencoder


👉 Implement Step 7: Response Generation

In [39]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


7.1 Load the LLM

In [40]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\aksha\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aksha\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back 

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [41]:
def build_prompt(question, retrieved_chunks, max_context_tokens=1200):
    header = (
        "Answer the question using ONLY the context below.\n"
        "If the answer is not present in the context, say \"I don't know\".\n\n"
        "Context:\n"
    )

    context = ""
    for i, ch in enumerate(retrieved_chunks):
        snippet = f"[{i+1}] {ch['text']}\n"
        context += snippet

    prompt = f"{header}{context}\nQuestion: {question}\nAnswer:"
    return prompt


In [53]:
import torch

def generate_answer(question, rrf_results, max_new_tokens=150):
    prompt = build_prompt(question, rrf_results)
    # print(prompt[:1500])

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,     # deterministic
            num_beams=4
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


In [54]:
def rag_answer(
    question,
    dense_model,
    faiss_index,
    faiss_metadata,
    bm25,
    chunks,
    top_k=10,
    top_n=5
):
    # Hybrid retrieval (Step 6)
    rrf_results = hybrid_retrieve(
        query=question,
        dense_model=dense_model,
        faiss_index=faiss_index,
        faiss_metadata=faiss_metadata,
        bm25=bm25,
        chunks=chunks,
        top_k=top_k,
        top_n=top_n
    )

    # Generation (Step 7)
    answer = generate_answer(question, rrf_results)

    return {
        "question": question,
        "answer": answer,
        "sources": rrf_results
    }


In [57]:
query = "what is bbc?"

result = rag_answer(
    question=query,
    dense_model=model,
    faiss_index=index,
    faiss_metadata=faiss_metadata,
    bm25=bm25,
    chunks=chunks,
    top_k=10,
    top_n=8
)

print("ANSWER:\n", result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print("-", s["title"], "| RRF:", round(s["rrf_score"], 4))


ANSWER:
 bbc news is responsible for the news programmes and documentary content on the bbc 's general television channels, as well as the news coverage on the bbc news channel in the uk, and 22 hours of programming for the corporation 's international bbc world news channel. coverage for bbc parliament is carried out on behalf of the bbc at millbank studios, though bbc news provides editorial and journalistic content. bbc news content is also output onto the bbc 's digital interactive television services under the bbc red button brand, and until 2012, on the ceefax teletext system. the

SOURCES:
- BBC News | RRF: 0.0328
- BBC News | RRF: 0.032
- BBC News | RRF: 0.0301
- BBC News | RRF: 0.0299
- BBC News | RRF: 0.0294
- BBC News | RRF: 0.029
- BBC News | RRF: 0.0161
- BBC News | RRF: 0.0159
